# Task 3 — On-Device Efficiency Benchmarking and Quantization Trade-offs

**Owner:** Ali Cherri  
**Hypothesis (H3):** W4A8 quantization will cut end-to-end latency by ~35% with less than 3 pp accuracy loss.

**Variations:** ShowUI-2B, OS-Atlas-4B, Ferret-UI Lite-3B, best Qwen2.5-VL-3B (LoRA), best PaliGemma-3B (LoRA), each at fp16 / bnb-int8 / bnb-4bit (NF4).

**Deeper analysis:** profiling time spent in vision encoding vs. decoding; compression robustness (which models degrade most under low-bit inference, and on which benchmark).

In [ ]:
import pandas as pd

from ais5.bench import run_full_benchmark, pareto_front, ParetoPoint
from ais5.bench.memory import vram_budget_check
from ais5.data import load_benchmark
from ais5.models import get_model
from ais5.quant import resolve_quant_config
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

In [ ]:
# Free Colab/Kaggle GPUs are 16GB; we enforce the 8GB on-device budget by
# measuring peak_vram_gb and rejecting cells that exceed it.
import torch
if torch.cuda.is_available():
    # Optional: cap VRAM at 8GB to make the budget binding rather than advisory.
    # torch.cuda.set_per_process_memory_fraction(8.0 / torch.cuda.get_device_properties(0).total_memory * (1024**3))
    print(torch.cuda.get_device_properties(0))

In [ ]:
MODELS = ['qwen2.5-vl-3b', 'paligemma-3b', 'showui-2b', 'os-atlas-4b']
QUANTS = ['none', 'bnb8', 'bnb4']
BENCHES = ['screenspot-v2', 'screenspot-pro', 'osworld-g']
LIMIT = 50  # set to None for the full benchmark

In [ ]:
rows = []
for model_name in MODELS:
    for quant_spec in QUANTS:
        qc = resolve_quant_config(quant_spec)
        kwargs = {'device_map': 'auto'}
        hf_quant = qc.to_hf()
        if hf_quant is not None:
            kwargs['quant_config'] = hf_quant
        try:
            model = get_model(model_name, **kwargs)
        except Exception as e:
            print(f'SKIP {model_name}|{qc.name}: {e}')
            continue
        for bench in BENCHES:
            samples = list(load_benchmark(bench))
            r = run_full_benchmark(model, samples, benchmark=bench, quant_label=qc.name, limit=LIMIT)
            rows.append(r.to_dict())
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
df = pd.DataFrame(rows)
df

## Pareto fronts

Two views: latency vs. accuracy and VRAM vs. accuracy. The non-dominated set tells us which (model, quant) combinations are worth deploying.

In [ ]:
import matplotlib.pyplot as plt

for cost_col, ax_label in [('latency_mean_ms', 'mean latency (ms)'), ('peak_vram_gb', 'peak VRAM (GB)')]:
    pts = [
        ParetoPoint(label=f"{r['model']}|{r['quant']}", accuracy=r['accuracy'], cost=r[cost_col])
        for r in rows if r['benchmark'] == 'screenspot-v2'
    ]
    front = pareto_front(pts)
    fig, ax = plt.subplots()
    ax.scatter([p.cost for p in pts], [p.accuracy for p in pts], alpha=0.4)
    ax.plot([p.cost for p in front], [p.accuracy for p in front], 'r-o', label='Pareto front')
    for p in front:
        ax.annotate(p.label, (p.cost, p.accuracy), fontsize=8)
    ax.set_xlabel(ax_label); ax.set_ylabel('accuracy'); ax.set_title(f'{ax_label} vs. accuracy — ScreenSpot-V2')
    ax.legend(); plt.show()

## Compression robustness

How much does each model lose when going from fp16 → 8-bit → 4-bit? Drop > 3 pp is flagged as failing H3.

In [ ]:
pivot = df.pivot_table(index=['model','benchmark'], columns='quant', values='accuracy')
pivot['drop_4bit'] = pivot.get('fp16', 0) - pivot.get('bnb-4bit-nf4', 0)
pivot